# Friends Recommendation (mutual-friend counting)

**Company:** MongoDB (GothamLoop question bank) · **Category:** Coding · **Tags:** Live Screen, Graphs, Hash Tables · **Difficulty/Frequency:** Rare (2/10)

## Concepts

**What this problem is really testing:**
- **Searching outward from the answer's neighbourhood**, instead of filtering the whole graph
- Counting **paths of length 2** in a graph — which is what "mutual friend" literally means
- Deterministic **tie-breaking**, and the `>` vs `>=` detail that decides it

**First-principles primer — what is each piece?**

- **The graph.** `{user: [friends]}` is an **adjacency list**. Users are vertices, friendships are edges. Nothing more exotic is needed.
- **A mutual friend is a path of length 2.** `u — f — c` says `u` knows `f`, and `f` knows `c`. So *"how many mutual friends do `u` and `c` have?"* is exactly *"how many 2-step paths run from `u` to `c`?"* Once you see it that way, the algorithm writes itself: walk two steps out from `u` and tally where you land.
- **Why not scan every user?** The naive version tests all N users and asks "how many friends do we share?". But **anyone with zero mutual friends can never be the answer**, and they are the overwhelming majority. Walking outward from `u` visits *only* the candidates that could possibly win.

**The complexity that follows:**

| Approach | Cost |
|---|---|
| Check every user | O(N × F) — N users, each compared against `u`'s F friends |
| Walk two steps out | **O(sum of the degrees of `u`'s friends)** |

In a real network the second is dramatically smaller: `u` has 200 friends who each have 200 friends, so you touch 40,000 nodes — not the 3 billion in the graph.

**The tie-break, and why `>` matters:**

```python
for candidate in sorted(mutual_counts):       # ascending ID
    if mutual_counts[candidate] > best_count: # STRICTLY greater
        best_count, best = mutual_counts[candidate], candidate
```

Because IDs are visited in ascending order, the first candidate to reach a given count claims it, and a **strict** `>` means a later (larger) ID can never displace it on a tie. Change that to `>=` and the rule silently inverts to *largest* ID wins. One character, opposite behaviour.

**Simple worked example.**

```
graph = {1: [2, 3],        u = 1
         2: [1, 4, 5],
         3: [1, 4],
         4: [2, 3, 5],
         5: [2, 4]}
```

`u = 1`'s friends are `{2, 3}`. Walk one step further from each:

| via friend | reaches | keep? |
|---|---|---|
| 2 | 1, 4, 5 | skip 1 (self); count **4**, **5** |
| 3 | 1, 4 | skip 1 (self); count **4** |

Counts: `{4: 2, 5: 1}`. **Answer: user 4**, with two mutual friends (2 and 3).

## Problem Statement

Given `{user_id: [friend_ids]}` and a target user, recommend the **best new friend**:

1. Most **mutual friends** with the target.
2. Ties broken by **smaller ID**.
3. Never recommend the target themselves, nor an existing friend.

```python
recommend_friend({1: [2, 3], 2: [1, 4, 5], 3: [1, 4],
                  4: [2, 3, 5], 5: [2, 4]}, user=1)
# -> 4   (shares friends 2 and 3)
```

### Approach 1 — Naive (test every user in the graph)

**Idea:** for each user in the graph, skip the target and its friends, then intersect their friend list with the target's to count mutuals.

Correct, and the honest baseline. The waste is that **almost everyone scores zero** — in a network of millions, only a few thousand share any friend with `u` at all, and this checks all of them.

**Time complexity:** **O(N × F)** — every user, intersected against the target's F friends.

**Space complexity:** O(F + C).

In [ ]:
from collections import defaultdict
from typing import Dict, List, Optional, Set


def recommend_friend_naive(graph: Dict[int, List[int]], user: int) -> int:
    friends = set(graph.get(user, []))
    best, best_count = -1, 0
    for candidate in sorted(graph):                 # EVERY user in the graph
        if candidate == user or candidate in friends:
            continue
        shared = len(friends & set(graph.get(candidate, [])))
        if shared > best_count:                     # strict >: ties keep the smaller ID
            best, best_count = candidate, shared
    return best

### Approach 2 — Optimal (walk two steps out and tally)

**Idea:** start at `u`, step to each friend, step again, and count where you land. Every landing is a mutual friend by construction — you got there *via* one of `u`'s friends.

**Three details worth defending:**

- **`friends` is a `set`, not a list.** The skip test `candidate in friends` runs once per edge examined. As a list that is O(F) each time, quietly making the whole thing O(F × E); as a set it is O(1).
- **Skip both `u` and existing friends *inside* the loop.** Filtering afterwards would work, but doing it here keeps the count map holding only genuine candidates, so the final scan has nothing to discard.
- **The count is incremented once per shared friend, not once per candidate.** That is the whole measurement: a candidate reached via three different friends of `u` scores 3.

**Time complexity:** **O(Σ deg(f) for f in friends(u))** — you touch each edge leaving `u`'s friends exactly once. Plus O(C log C) for the sorted tie-break scan.

**Space complexity:** O(F + C).

In [ ]:
def recommend_friend(graph: Dict[int, List[int]], user: int) -> int:
    friends = set(graph.get(user, []))              # a SET: the skip test below is O(1)
    mutual_counts: Dict[int, int] = defaultdict(int)

    for friend in friends:                          # step 1: u -> f
        for candidate in graph.get(friend, []):     # step 2: f -> c
            if candidate == user or candidate in friends:
                continue                            # never recommend self or an existing friend
            mutual_counts[candidate] += 1           # one increment PER SHARED FRIEND

    if not mutual_counts:
        return -1                                   # nobody is two steps away

    best, best_count = -1, -1
    for candidate in sorted(mutual_counts):         # ascending ID...
        if mutual_counts[candidate] > best_count:   # ...and STRICT >, so ties keep the smaller
            best, best_count = candidate, mutual_counts[candidate]
    return best

### Approach 3 — Handling asymmetric graphs, top-k, and weights

**Idea:** three extensions the follow-ups ask for, each a small change to the same tally.

- **`symmetric=True`.** The official code counts a candidate whenever *some friend of `u` lists them* — it never checks the friendship is mutual. If `A` lists `B` but `B` does not list `A`, that one-way edge still counts. Normalising the graph first (adding every reverse edge) makes "friend" mean what the word means. Worth asking about: an asymmetric adjacency map is usually a data bug, but sometimes it is a *follow* relationship, where one-way is correct.
- **Top-k.** Sort by `(-count, id)` — descending count, ascending ID — which is the same tie-break rule expressed as a sort key. For large candidate sets, `heapq.nsmallest(k, ...)` on that key is O(C log k).
- **Weights.** Replace `+= 1` with `+= weight[friend]`. This is where real recommenders live: a mutual friend who is themselves highly connected is weak evidence (they know everyone), so production systems down-weight by degree — `1 / log(deg(f))` is the classic **Adamic–Adar** score.

**Time complexity:** unchanged, plus O(V + E) for normalisation.

**Space complexity:** O(V + E) for the normalised copy.

In [ ]:
import heapq
import math


def normalise(graph: Dict[int, List[int]]) -> Dict[int, Set[int]]:
    """Make every edge bidirectional, and drop duplicate/self edges."""
    sym: Dict[int, Set[int]] = defaultdict(set)
    for u, friends in graph.items():
        sym.setdefault(u, set())
        for v in friends:
            if u == v:
                continue                            # a self-loop is not a friendship
            sym[u].add(v)
            sym[v].add(u)                           # the edge the input may have omitted
    return sym


def mutual_counts(graph, user: int, symmetric: bool = False,
                  weighted: bool = False) -> Dict[int, float]:
    g = normalise(graph) if symmetric else {k: list(v) for k, v in graph.items()}
    friends = set(g.get(user, ()))
    counts: Dict[int, float] = defaultdict(float)
    for f in friends:
        # Adamic-Adar: a friend who knows everyone is WEAK evidence, so weight by 1/log(degree).
        w = 1.0 / math.log(len(g.get(f, ())) + 1e-9) if weighted and len(g.get(f, ())) > 1 else 1.0
        for c in g.get(f, ()):
            if c == user or c in friends:
                continue
            counts[c] += w
    return counts


def recommend_top_k(graph, user: int, k: int = 3, symmetric: bool = False,
                    weighted: bool = False) -> List[int]:
    counts = mutual_counts(graph, user, symmetric, weighted)
    # (-count, id): highest count first, then smallest id - the tie-break as a sort key.
    return [c for _, c in heapq.nsmallest(k, ((-n, c) for c, n in counts.items()))]


def recommend_friend_ex(graph, user: int, symmetric: bool = False,
                        weighted: bool = False) -> int:
    top = recommend_top_k(graph, user, 1, symmetric, weighted)
    return top[0] if top else -1

## Verification

The worked example, the tie-break rule in both directions, the exclusion rules, and an exhaustive cross-check against the naive implementation on random graphs.

In [ ]:
import random

GRAPH = {1: [2, 3], 2: [1, 4, 5], 3: [1, 4], 4: [2, 3, 5], 5: [2, 4]}

# --- The worked example ---
for fn in (recommend_friend, recommend_friend_naive, recommend_friend_ex):
    assert fn(GRAPH, 1) == 4, f"{fn.__name__}: 4 shares friends 2 and 3"
assert dict(mutual_counts(GRAPH, 1)) == {4: 2, 5: 1}

# --- The exclusion rules ---
for fn in (recommend_friend, recommend_friend_naive):
    r = fn(GRAPH, 1)
    assert r != 1, f"{fn.__name__}: never recommend the user themselves"
    assert r not in GRAPH[1], f"{fn.__name__}: never recommend an existing friend"

# --- THE tie-break: equal counts must pick the SMALLER id ---
tie = {1: [10, 20],
       10: [1, 50, 99],       # 50 and 99 each get one mutual friend...
       20: [1, 50, 99],       # ...and now each has two. A perfect tie.
       50: [10, 20], 99: [10, 20]}
assert dict(mutual_counts(tie, 1)) == {50: 2, 99: 2}, "a genuine tie"
for fn in (recommend_friend, recommend_friend_naive, recommend_friend_ex):
    assert fn(tie, 1) == 50, f"{fn.__name__}: on a tie, the SMALLER id wins"

# The rule must hold whichever order the ids happen to be visited in
reversed_tie = {1: [20, 10], 10: [1, 99, 50], 20: [1, 99, 50],
                50: [10, 20], 99: [10, 20]}
assert recommend_friend(reversed_tie, 1) == 50, "insertion order must not affect the result"

# A higher count always beats a smaller id
beats = {1: [10, 20], 10: [1, 99], 20: [1, 99, 50], 50: [20], 99: [10, 20]}
assert dict(mutual_counts(beats, 1)) == {99: 2, 50: 1}
assert recommend_friend(beats, 1) == 99, "count dominates; the id is only a tie-break"

# --- No recommendation possible ---
assert recommend_friend({1: []}, 1) == -1, "the user has no friends"
assert recommend_friend({}, 1) == -1, "the user is not in the graph at all"
assert recommend_friend({1: [2], 2: [1]}, 1) == -1, (
    "the only friend knows nobody new"
)
assert recommend_friend({1: [2, 3], 2: [1, 3], 3: [1, 2]}, 1) == -1, (
    "a closed triangle: everyone reachable is already a friend"
)

# --- A friend of a friend who is ALSO already a friend must not be counted ---
overlap = {1: [2, 3], 2: [1, 3, 7], 3: [1, 2, 7], 7: [2, 3]}
counts = dict(mutual_counts(overlap, 1))
assert 2 not in counts and 3 not in counts, "existing friends are excluded from the tally"
assert counts == {7: 2}
assert recommend_friend(overlap, 1) == 7

# --- Asymmetric graphs: the two readings genuinely differ ---
asym = {1: [2],
        2: [1, 9],
        9: []}          # 9 does NOT list 2 back
assert recommend_friend(asym, 1) == 9, "the raw reading counts the one-way edge"
assert recommend_friend_ex(asym, 1, symmetric=False) == 9
assert recommend_friend_ex(asym, 1, symmetric=True) == 9, (
    "normalising ADDS 9->2, so 9 is still a candidate"
)

# The reverse case: the edge the raw reading MISSES
asym2 = {1: [2], 2: [1], 9: [2]}      # 9 lists 2, but 2 does not list 9
assert recommend_friend(asym2, 1) == -1, "the raw reading never sees 9, since 2 does not list it"
assert recommend_friend_ex(asym2, 1, symmetric=True) == 9, (
    "normalising adds 2->9, so 9 becomes reachable - a DIFFERENT answer"
)

# --- Duplicate and self edges would otherwise inflate the count ---
dupes = {1: [2, 2], 2: [1, 1, 8, 8], 8: [2]}
assert dict(mutual_counts(dupes, 1))[8] == 2, "duplicates inflate the raw count"
assert dict(mutual_counts(dupes, 1, symmetric=True))[8] == 1, (
    "normalising to sets removes the double-count"
)
selfloop = {1: [1, 2], 2: [1, 5], 5: [2]}
assert 1 not in mutual_counts(selfloop, 1), "the user is never their own recommendation"

# --- Top-k ---
wide = {1: [2, 3, 4],
        2: [1, 10, 11, 12],
        3: [1, 10, 11],
        4: [1, 10],
        10: [2, 3, 4], 11: [2, 3], 12: [2]}
assert dict(mutual_counts(wide, 1)) == {10: 3, 11: 2, 12: 1}
assert recommend_top_k(wide, 1, k=3) == [10, 11, 12], "descending count"
assert recommend_top_k(wide, 1, k=2) == [10, 11]
assert recommend_top_k(wide, 1, k=99) == [10, 11, 12], "k larger than the candidate set"
assert recommend_top_k({1: []}, 1, k=3) == []
# Top-k must obey the same tie-break
assert recommend_top_k(tie, 1, k=2) == [50, 99], "tied counts ordered by ascending id"

# --- Weighting: a hyper-connected mutual friend is weaker evidence ---
hub = {1: [2, 3],
       2: [1] + list(range(100, 200)) + [500],   # 2 knows everyone - weak signal
       3: [1, 500],                              # 3 is selective - strong signal
       500: [2, 3]}
assert dict(mutual_counts(hub, 1))[500] == 2
w = mutual_counts(hub, 1, weighted=True)
assert w[500] < 2, "weighting discounts the hub's endorsement"
assert w[500] > w[150], "500 is endorsed by BOTH, 150 only by the hub"

# --- Exhaustive cross-check against the naive implementation ---
random.seed(101)
for _ in range(500):
    n = random.randint(2, 12)
    users = list(range(n))
    g: Dict[int, List[int]] = {u: [] for u in users}
    for a in users:                              # build a SYMMETRIC random graph
        for b in users:
            if a < b and random.random() < 0.3:
                g[a].append(b)
                g[b].append(a)
    for u in users:
        expected = recommend_friend_naive(g, u)
        assert recommend_friend(g, u) == expected, (g, u)
        assert recommend_friend_ex(g, u) == expected, (g, u)
        # A returned recommendation must always satisfy every stated rule
        r = recommend_friend(g, u)
        if r != -1:
            assert r != u and r not in g[u]
            assert len(set(g[u]) & set(g[r])) > 0, "a recommendation must share a friend"

print("All assertions passed.")

## Discussion — remaining follow-up directions

- **Asymmetric storage.** Implemented as `symmetric=True`, and the tests show it changing the answer in **both** directions: a one-way edge that the raw reading wrongly counts, and one it wrongly misses. Worth asking about rather than assuming — an asymmetric adjacency map is usually a data bug, but sometimes it is deliberate, because a *follow* is genuinely one-way. "Mutual friends" and "people who follow the same people" are different metrics, and the code cannot tell you which was meant.
- **Top-k.** `sorted(key=(-count, id))`, or `heapq.nsmallest(k, ...)` on the same key for O(C log k) — the same size-k heap idea as [Scores Finding](../19.%20Scores_Finding/19.%20Scores_Finding.ipynb). The detail that matters is that the tie-break rule must be **the same** in both code paths; expressing it once as a sort key rather than twice as comparison logic is what keeps them consistent.
- **Weighting by closeness.** `+= 1` becomes `+= w(f)`. The principled choice is **Adamic–Adar**: weight each shared friend by `1 / log(degree)`, because a mutual friend who knows 10,000 people tells you almost nothing, while one who knows 12 is a strong signal. The `hub` test above shows it working. This is the step from "counting" to "recommending".
- **A graph too large for memory.** The two-step walk is exactly a **map-reduce**: map each `(friend, candidate)` pair to `(candidate, 1)` and reduce by summing. It parallelises perfectly because each of `u`'s friends is processed independently. The real difficulty is not the algorithm but **skew** — a celebrity with 50 million followers makes one reducer key dwarf all the others, and the standard fix is to cap or specially-handle very high-degree nodes.
- **Single pass without the count map.** Possible, but the follow-up's own caveat is the point: a candidate is encountered *multiple times*, so you cannot decide the winner until you have seen every increment. You would still need the counts — you would just be tracking the running best alongside them, saving the final sort rather than the map. That is O(C) instead of O(C log C), for meaningfully more fiddly tie-break logic.
- **Why this is only a starting point.** Counting mutual friends is a structural signal and nothing more. Real recommenders blend it with interaction recency, shared groups, geography, and explicit negative feedback ("don't show me this person") — and they deliberately **exclude** some high-mutual-count candidates, because the person you share the most friends with may be someone you have gone out of your way to avoid.

## Empirical complexity check

Compare **scanning every user** with **walking two steps out**, on a sparse random graph where each user has a fixed average degree and the user count doubles.

| Growth when the user count doubles | What it means |
|---|---|
| ~2x | linear in N — every user is tested, however hopeless |
| ~1x | independent of N — the work depends only on the neighbourhood |

That flat row is the whole point: in a real social network, the number of people within two hops of you does not grow when the network adds a million strangers.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

import random

QUERIES = 200
DEGREE = 8          # average friends per user - FIXED, so the neighbourhood does not grow


def make_graph(n):
    rng = random.Random(103)
    g: Dict[int, List[int]] = {u: [] for u in range(n)}
    for u in range(n):
        for _ in range(DEGREE // 2):
            v = rng.randrange(n)
            if v != u:
                g[u].append(v)
                g[v].append(u)
    users = [rng.randrange(n) for _ in range(QUERIES)]
    return (g, users)


def run_naive(g, users):
    for u in users:
        recommend_friend_naive(g, u)        # O(N) per query - scans every user


def run_walk(g, users):
    for u in users:
        recommend_friend(g, u)              # O(neighbourhood) per query


benchmark(
    {"Approach 1 - test every user O(N*F)": run_naive,
     "Approach 2 - walk two steps out": run_walk},
    make_graph,
    sizes=[1000, 2000, 4000, 8000],
    repeats=2,
)

## Patterns learned

- **Search outward from the answer, not inward from the universe.** Only people within two hops can possibly share a friend with you. Enumerating all N users and filtering is the difference between O(N) and O(neighbourhood) — and in a real graph that is a factor of millions.
- **"Mutual friend" is a path of length 2.** Naming the graph-theoretic shape turns a vague social question into a concrete traversal: step out, step out again, tally where you land.
- **Membership tests belong in a set.** `candidate in friends` runs once per edge; as a list that is a hidden O(F) factor that turns a linear algorithm quadratic.
- **Express a tie-break once, as a sort key.** `(-count, id)` is the whole rule. Written as comparison logic instead, it has to be repeated in the top-1 and top-k paths, and the two will eventually disagree.
- **`>` versus `>=` decides which side of a tie wins.** With candidates visited in ascending order, strict `>` keeps the smaller id. One character, inverted behaviour, and no test will catch it unless you write one that ties.
- **Normalise the graph before trusting its shape.** Asymmetric, duplicate and self edges all silently corrupt a count. Whether asymmetry is a bug or a *follow* relationship is a question for the interviewer, not an assumption.
- **Counting is not yet recommending.** Weighting shared friends by `1 / log(degree)` — Adamic–Adar — captures the real insight: an endorsement from someone who knows everyone is worth almost nothing.